In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# ── Configuración del modelo ──────────────────────────────
token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.1,
    streaming=True
)

# ── Configurar prompt con memoria ────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente profesional en el area automotriz, mas destinada al vulcanizado es decir parchado de neumaticos y montados de neumaticos, tambien tipo de llantas y radio, con gran conocimiento del trabajo."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm

# ── Almacén de historial por sesión ──────────────────────
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# ── Envolver con memoria ──────────────────────────────────
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# ── Función de chat ───────────────────────────────────────
session_id = "mi_sesion"

def chat(mensaje):
    response = conversation.invoke(
        {"input": mensaje},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"Cliente: {mensaje}")
    print(f"Profesional Vulcanizador: {response.content}")
    print("-" * 40)

# ── Prueba — la IA debe recordar el contexto ─────────────
chat("Hola! Me llamo Jesus.")
chat("me gustaria saber que puedo hacer con un neumatico con los alambres salidos tambien tiene una forma de huevo deberia parchar o cambiar?")
chat("recuerdas mi nombre?")  # ← Esta prueba confirma que la memoria funciona

Cliente: Hola! Me llamo Jesus.
Profesional Vulcanizador: ¡Hola, Jesús! ¿En qué puedo ayudarte hoy en el área de neumáticos y vulcanizado?
----------------------------------------
Cliente: me gustaria saber que puedo hacer con un neumatico con los alambres salidos tambien tiene una forma de huevo deberia parchar o cambiar?
Profesional Vulcanizador: Cuando un neumático presenta alambres salidos y tiene una forma de huevo (lo que indica una deformación o abultamiento), es una señal clara de que el neumático está dañado de manera significativa. Este tipo de daño generalmente se debe a un golpe fuerte, desgaste irregular o problemas de presión.

En este caso, lo más recomendable es **cambiar el neumático**. Intentar parchar un neumático con alambres expuestos y deformaciones puede ser muy peligroso, ya que puede comprometer la integridad estructural del neumático y aumentar el riesgo de un reventón mientras conduces.

Siempre es mejor priorizar la seguridad. Si tienes dudas sobre el estado 